# RetailMind AI — Day 1: EDA & Churn Baseline

This notebook mirrors the production pipeline in `src/`. Use it to explore, then run `python -m src.train` to persist the model.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import DECISION_THRESHOLD, RANDOM_STATE, TEST_SIZE
from src.data import clean, load_raw, split_features_target
from src.evaluate import evaluate_model
from src.pipeline import build_pipeline

## 1. Ingest and inspect

In [ ]:
raw = load_raw()
print(f"rows={len(raw):,} columns={raw.shape[1]}")
raw.head()

In [ ]:
schema = pd.DataFrame({"dtype": raw.dtypes.astype(str)})
display(schema)

blank_total_charges = raw["TotalCharges"].astype(str).str.strip().eq("").sum()
print("Blank TotalCharges values:", blank_total_charges)

balance = raw["Churn"].value_counts(normalize=True).round(4)
print("\nTarget balance:")
display(balance)
print("Churn rate: {:.2%}".format(balance["Yes"]))

## 2. Split, preprocess and train

`clean()` coerces `TotalCharges` to numeric and fills the 11 blanks with `0.0` (new customers, `tenure = 0`), then maps the target to 0/1.

In [ ]:
df = clean(raw)
X, y = split_features_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"train={len(X_train):,} test={len(X_test):,} features={X.shape[1]}")

In [ ]:
pipeline = build_pipeline(feature_columns=list(X.columns))
pipeline.fit(X_train, y_train)

## 3. Evaluate

In [ ]:
metrics, predictions, probabilities = evaluate_model(
    pipeline, X_test, y_test, threshold=DECISION_THRESHOLD
)

pd.DataFrame(
    {key: [metrics[key]] for key in ("accuracy", "precision", "recall", "f1", "roc_auc")},
    index=["score"],
).T

In [ ]:
print(metrics["classification_report"])
print("Confusion matrix:", metrics["confusion_matrix"])

## 4. Persist and serve

Run the production entry point once to write `models/churn_pipeline.joblib` and `models/metrics.json`, then score customers with `src.service.predict()`.

In [ ]:
# !python -m src.train
from src.service import predict

predict(dict(X_test.iloc[0]))